# ARCHS4 CLAMP models with all pathways prior at 50% sample coverage (Fixed K from 100%)

**Environment:** `clamp-analyses`

Runs CLAMPfull with the combined all-pathways prior (Hallmark, Reactome, GO CC, C8) at 50% sample coverage, reusing the subsampled FBM, SVD, and CLAMPbase results already generated in `06_bp_coverage_rs/04_bp_coverage_C2CP_rs_50.ipynb`.

**Key difference from `06_bp_coverage_rshall`:** CLAMP_K is taken from the 100% coverage model (`c2cp_coverage_rs100_seed_*`) rather than the 50% model, so the number of LVs is fixed to that inferred from the full dataset.

Steps (repeated for each seed):
1. Load existing subsampled FBM, SVD, and CLAMPbase from `c2cp_coverage_rs50_seed_*`
2. Load CLAMP_K from `c2cp_coverage_rs100_seed_*`
3. Run CLAMPfull with the combined all-pathways prior
4. Save results to `hall_fixedK_coverage_rs50_seed_*`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER

coverage     <- 0.50
coverage_pct <- coverage * 100

data_path     <- here::here('data/archs4')
pathways_path <- here::here('data/pathways')

MULTIPLIER <- 100
MAX_ITER   <- 5000
n_runs     <- 3

message("Coverage: ", coverage_pct, "% — will run ", n_runs, " seeds")
message("CLAMP_K will be loaded from 100% coverage model")

## Load metadata and all pathways prior

In [ ]:
meta          <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin  <- meta$n_genes_thin
archs4_genes  <- meta$gene_symbols_thin

hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, archs4_genes)
message("Loaded and matched all pathways matrix")

## Run CLAMPbase and CLAMPfull with all pathways prior + fixed K (3 seeds)

In [ ]:
for (run_idx in seq_len(n_runs)) {
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs)
  message(strrep("=", 60))

  src_dir  <- file.path(base_output_dir,
    paste0("c2cp_coverage_rs", coverage_pct, "_seed_", run_idx))
  k100_dir <- file.path(base_output_dir, "c2cp_coverage_rs100_seed_1")
  dst_dir  <- file.path(base_output_dir,
    paste0("hall_fixedK_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

  # Load existing artifacts from the per-coverage directory
  subsample_info <- readRDS(file.path(src_dir, "subsample_info.rds"))
  sample_names   <- subsample_info$sample_names
  n_samples      <- subsample_info$n_samples

  svd_result <- readRDS(file.path(src_dir, "svd.rds"))

  # Load CLAMP_K from the 100% coverage model
  CLAMP_K <- readRDS(file.path(k100_dir, "CLAMP_K.rds"))
  message("Using CLAMP_K = ", CLAMP_K, " (from 100% model, seed 1)")

  Y_sub <- FBM(
    nrow        = n_genes_thin,
    ncol        = n_samples,
    backingfile = file.path(src_dir, "fbm_subsampled"),
    create_bk   = FALSE
  )

  # CLAMPbase with fixed K
  message("Running CLAMPbase with fixed K...")
  baseRes <- CLAMPbase(
    Y        = Y_sub,
    svdres   = svd_result,
    trace    = TRUE,
    clamp_k  = CLAMP_K
  )

  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names

  saveRDS(baseRes, file = file.path(dst_dir, "CLAMPbase.rds"))

  model_dir <- file.path(dst_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

  # CLAMPfull with all pathways prior and fixed K
  message("Running CLAMPfull with all pathways prior...")
  fullRes <- CLAMPfull(
    Y                 = Y_sub,
    svdres            = svd_result,
    priorMat          = all_pathways_matched,
    clamp.base.result = baseRes,
    use_cpp           = TRUE,
    trace             = TRUE,
    multiplier        = MULTIPLIER,
    max.iter          = MAX_ITER,
    clamp_k           = CLAMP_K
  )

  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))

  saveRDS(fullRes, file = file.path(dst_dir, "CLAMPfull_hall.rds"))

  model_dir <- file.path(dst_dir, "CLAMPfull_hall")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

  rm(Y_sub, svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))